In [ ]:
from llama_index.core import KnowledgeGraphIndex
from llama_index.core import StorageContext, load_index_from_storage
from llama_index.core.graph_stores import SimpleGraphStore
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.core import Settings
embed_model = HuggingFaceEmbedding(model_name="BAAI/bge-small-en-v1.5")
Settings.embed_model = embed_model
Settings.llm = None
Settings.chunk_size = 512

In [ ]:
import json
with open('parsed_balakanda_chapter_1_triplets.json', 'r') as file:
    parsed_data = json.load(file)

file = 'balakanda_chapter1.json'
import json
with open(file) as f:
    data = json.load(f)

from llama_index.core import Document
documents = []
for i in data[:]:       
    document = Document(
    text = i['content'],
    metadata={
        "canto": i['chapter'],
        "canto_summary": i['summary'],
        "r_id": str(i['chapter']) + "_" + str(i['prompt_number'])
    },
    excluded_llm_metadata_keys=["r_id"],
    metadata_seperator="::",
    metadata_template="{key}=>{value}",
    text_template="Content: {content}, Extra Chapter context for reference: {metadata_str}\n",
    )
    documents.append(document)
print(len(documents), len(parsed_data))
print(f"Document Metadata: {documents[-1].metadata}")

In [ ]:
from llama_index.core.node_parser import SentenceSplitter

node_parser = SentenceSplitter()
nodes = node_parser.get_nodes_from_documents(documents)

index = load_index_from_storage(StorageContext.from_defaults(persist_dir=r"C:\Users\M LAKSHMI NARASIMHA\OneDrive\Desktop\Ramayan\KGs\balakanda"))

from tqdm import tqdm
for i in tqdm(range(0, len(nodes))):
    if i > 0 and i % 30 == 0:
        index.storage_context.persist(persist_dir='balakanda_chapter1_update_temp')
    for key,value in parsed_data[i].items():
        for tup in value:
            index.upsert_triplet_and_node(tuple(tup), nodes[i], include_embeddings=True)

index.storage_context.persist(persist_dir='balakanda_chapter1_update')

In [ ]:
query_engine1 = index.as_query_engine(
    include_text=True, 
    response_mode="tree_summarize",
    embedding_mode="hybrid",
    similarity_top_k=5,
    verbose=True)
response = query_engine1.query("Name rama's bow?")

In [ ]:
from llama_index.core import StorageContext, load_index_from_storage
from custom_llm import OurLLM
from llama_index.core import ServiceContext, set_global_service_context
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
llmR = OurLLM()
embed_model = HuggingFaceEmbedding(model_name="BAAI/bge-small-en-v1.5")
set_global_service_context(ServiceContext.from_defaults(llm=llmR, embed_model=embed_model, chunk_size=2048, context_window = 256, num_output=256))

index = load_index_from_storage(StorageContext.from_defaults(persist_dir=r"C:\Users\M LAKSHMI NARASIMHA\OneDrive\Desktop\Ramayan\KGs\balakanda"))
query_engine1 = index.as_query_engine(
    include_text=True, 
    response_mode="tree_summarize",
    embedding_mode="hybrid",
    similarity_top_k=5,
    verbose=True)
response = query_engine1.query("Name rama's bow that belonged to indra?")
print(response)